# Fintech Privacy Filter — Training & Evaluation Notebook

Fine-tunes `openai/privacy-filter` on `msdakot/fintech-privacy-pii` (65 fintech labels), then evaluates per-label and per-language F1 on the held-out test set.

**Runtime:** Training ~6–8 hrs on T4 + High RAM, evaluation ~30 min after training completes.

> **Keep your browser open for the full run.** `opf train` writes the checkpoint at end of training only — a mid-run disconnect loses progress. If you need background execution, upgrade to Colab Pro+.

---
### Checklist before running
- [ ] Runtime → Change runtime type → **T4 GPU** + **High RAM** (required — checkpoint write OOMs on standard RAM)
- [ ] Left sidebar → Key icon → Add secret `HF_TOKEN` (HuggingFace write-access token)
- [ ] Left sidebar → Key icon → Add secret `WANDB_API_KEY` (from wandb.ai/authorize)
- [ ] ~5 GB free on Google Drive
- [ ] Run cells top to bottom

## Preparation

In [ ]:
import json
import os
import subprocess
import sys
import time
from collections import defaultdict
from pathlib import Path

import wandb
from datasets import load_dataset
from google.colab import drive, userdata
from huggingface_hub import HfApi, create_repo, snapshot_download

In [ ]:
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU detected — switch runtime to T4 before continuing.')

In [ ]:
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/fintech-pii-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

In [ ]:
!pip install -q 'opf @ git+https://github.com/openai/privacy-filter.git' datasets huggingface_hub wandb
print('Dependencies installed.')

In [ ]:
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
user = HfApi(token=HF_TOKEN).whoami()
print(f"HF authenticated as: {user['name']}")

os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
wandb.login()
print('W&B authenticated.')

## Data & Model

In [ ]:
# opf downloads its own base model automatically to /root/.opf/privacy_filter
# on first run (~2.8 GB). No manual download needed — opf fetches the correct
# opf-format files using its own internal download path.
print('Base model will be fetched automatically by opf train.')

In [ ]:
DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(exist_ok=True)

def record_to_opf(record):
    """Convert HF dataset record → opf JSONL format.
    HF:  spans = [{start, end, label}, ...]
    opf: spans = {label: [[start, end], ...], ...}
    """
    spans_dict = {}
    for span in record.get('spans', []):
        label = span['label']
        spans_dict.setdefault(label, []).append([span['start'], span['end']])
    return {'text': record['text'], 'spans': spans_dict}

print('Loading msdakot/fintech-privacy-pii from HF Hub...')
ds = load_dataset('msdakot/fintech-privacy-pii', token=HF_TOKEN)

split_sizes = {}
for split in ('train', 'validation', 'test'):
    path = DATA_DIR / f'{split}.jsonl'
    with open(path, 'w', encoding='utf-8') as f:
        for record in ds[split]:
            f.write(json.dumps(record_to_opf(record), ensure_ascii=False) + '\n')
    split_sizes[split] = len(ds[split])
    print(f'  {split}: {len(ds[split])} records → {path}')

print('Export complete.')

# Per-language test subsets — used for language-level F1 eval
lang_records = defaultdict(list)
for record in ds['test']:
    lang = record.get('language', 'en')
    lang_records[lang].append(record)

lang_paths = {}
for lang, records in sorted(lang_records.items()):
    lang_path = DATA_DIR / f'test_{lang}.jsonl'
    with open(lang_path, 'w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record_to_opf(record), ensure_ascii=False) + '\n')
    lang_paths[lang] = str(lang_path)
    print(f'  test/{lang}: {len(records)} examples → {lang_path}')

In [ ]:
label_space = {
    'span_class_names': [
        'O',
        'account_number', 'age', 'api_key', 'bank_routing_number',
        'biometric_identifier', 'blood_type', 'certificate_license_number',
        'city', 'company_name', 'coordinate', 'country', 'county',
        'credit_debit_card', 'customer_id', 'cvv', 'date', 'date_of_birth',
        'date_time', 'device_identifier', 'education_level', 'email',
        'employee_id', 'employment_status', 'fax_number', 'first_name',
        'gender', 'health_plan_beneficiary_number', 'http_cookie',
        'ipv4', 'ipv6', 'language', 'last_name', 'license_plate',
        'mac_address', 'medical_record_number', 'national_id', 'occupation',
        'password', 'phone_number', 'pin', 'political_view', 'postcode',
        'race_ethnicity', 'religious_belief', 'sexuality', 'ssn', 'state',
        'street_address', 'swift_bic', 'tax_id', 'time', 'unique_id',
        'url', 'user_name', 'vehicle_identifier',
        'iban', 'bban', 'lei', 'isin', 'cusip',
        'swift_mt_ref', 'policy_number', 'vat_number', 'loan_number', 'crypto_address',
    ]
}

LABEL_SPACE_PATH = '/content/label_space.json'
with open(LABEL_SPACE_PATH, 'w') as f:
    json.dump(label_space, f, indent=2)

print(f'Label space: {len(label_space["span_class_names"]) - 1} entity types + O')

## Sanity Check

1 epoch on 200 examples — verifies data format, checkpoint, and label space before the full run. Fix errors here, not mid-training.

In [ ]:
SANITY_DIR = '/content/sanity-check'
os.makedirs(SANITY_DIR, exist_ok=True)

sanity_cmd = [
    'opf', 'train', str(DATA_DIR / 'train.jsonl'),
    '--validation-dataset', str(DATA_DIR / 'validation.jsonl'),
    '--label-space-json', LABEL_SPACE_PATH,
    '--output-dir', SANITY_DIR,
    '--overwrite-output',
    '--max-train-examples', '200',
    '--max-validation-examples', '50',
    '--epochs', '1',
    '--batch-size', '2',
    '--grad-accum-steps', '4',
    '--learning-rate', '1e-4',
    '--weight-decay', '0.0',
    '--output-param-dtype', 'bf16',
]

print('Running sanity check...')
result = subprocess.run(sanity_cmd)

if result.returncode != 0:
    raise RuntimeError('Sanity check failed — fix the error above before running full training.')

print('\nSanity check passed. Checkpoint files:', os.listdir(SANITY_DIR))
print('Proceed to full training.')

## Training

Initialises W&B then runs the full 3-epoch job (~2–4 hrs on T4). Keep the browser open — checkpoint is written at the end of training.

In [ ]:
run = wandb.init(
    project='fintech-privacy-filter',
    name='opf-base-fintech-v0',
    config={
        'base_model': 'openai/privacy-filter',
        'output_model': 'msdakot/fintech-privacy-filter-v0',
        'dataset': 'msdakot/fintech-privacy-pii',
        'train_examples': split_sizes['train'],
        'val_examples': split_sizes['validation'],
        'test_examples': split_sizes['test'],
        'languages': ['en', 'es', 'de', 'fr', 'it', 'nl', 'sv'],
        'n_entity_types': len(label_space['span_class_names']) - 1,
        'new_fintech_labels': 10,
        'learning_rate': 1e-4,
        'epochs': 3,
        'batch_size': 2,
        'grad_accum_steps': 4,
        'effective_batch_size': 8,
        'weight_decay': 0.0,
        'output_param_dtype': 'bf16',
        'hardware': 'T4',
    }
)

print(f'W&B run: {run.url}')

In [ ]:
cmd = [
    'opf', 'train', str(DATA_DIR / 'train.jsonl'),
    '--validation-dataset', str(DATA_DIR / 'validation.jsonl'),
    '--label-space-json', LABEL_SPACE_PATH,
    '--output-dir', CHECKPOINT_DIR,
    '--overwrite-output',
    '--epochs', '3',
    '--batch-size', '2',
    '--grad-accum-steps', '4',
    '--learning-rate', '1e-4',
    '--weight-decay', '0.0',
    '--output-param-dtype', 'bf16',
]

print('Command:', ' '.join(cmd))
print('Training started...')
print('-' * 60)

t0 = time.time()
result = subprocess.run(cmd)
elapsed = time.time() - t0

wandb.log({'training_duration_minutes': elapsed / 60})

if result.returncode != 0:
    wandb.finish(exit_code=result.returncode)
    raise RuntimeError(f'opf train failed with code {result.returncode}')

print(f'\nTraining complete in {elapsed/60:.1f} minutes.')

## Results

In [ ]:
print('Checkpoint contents:', os.listdir(CHECKPOINT_DIR))

summary_path = os.path.join(CHECKPOINT_DIR, 'finetune_summary.json')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print('\nTraining summary:')
    print(json.dumps(summary, indent=2))

    def _flatten(d, prefix=''):
        out = {}
        for k, v in d.items():
            key = f'{prefix}/{k}' if prefix else k
            if isinstance(v, dict):
                out.update(_flatten(v, key))
            elif isinstance(v, (int, float)):
                out[key] = v
        return out

    wandb.log(_flatten(summary, 'train'))
    print('\nSummary logged to W&B.')
else:
    print('WARNING: finetune_summary.json not found.')

In [ ]:
HF_MODEL_REPO = 'msdakot/fintech-privacy-filter-v0'
api = HfApi(token=HF_TOKEN)

create_repo(HF_MODEL_REPO, repo_type='model', exist_ok=True, token=HF_TOKEN)

api.upload_folder(
    folder_path=CHECKPOINT_DIR,
    repo_id=HF_MODEL_REPO,
    repo_type='model',
    commit_message='Initial fine-tuned checkpoint — fintech-privacy-filter-v0',
)

model_url = f'https://huggingface.co/{HF_MODEL_REPO}'
wandb.log({'hf_model_url': model_url})
print(f'Model pushed: {model_url}')

In [ ]:
local_model = snapshot_download(repo_id=HF_MODEL_REPO, token=HF_TOKEN)
print('Model downloaded to:', local_model)

test_texts = [
    'Wire transfer to IBAN DE89370400440532013000 completed.',
    'The LEI 529900T8BM49AURSDO55 was submitted to the trade repository.',
    'Contact john.smith@acmecorp.com for invoice queries.',
]

table = wandb.Table(columns=['input', 'redacted'])

for text in test_texts:
    res = subprocess.run(
        ['opf', 'redact', '--checkpoint', local_model, text],
        capture_output=True, text=True
    )
    redacted = res.stdout.strip()
    print(f'Input:    {text}')
    print(f'Redacted: {redacted}\n')
    table.add_data(text, redacted)

wandb.log({'inference_examples': table})

## Evaluation

Runs `opf eval` on the held-out test set — full dataset then per-language subsets.
Results are logged to the same W&B run and printed as copy-paste README tables.

In [ ]:
def _get_overall_f1(results: dict) -> float:
    """Extract overall F1 — handles common opf eval output formats."""
    if 'overall' in results:
        return results['overall'].get('f1', 0.0)
    if 'f1' in results:
        return results['f1']
    for v in results.values():
        if isinstance(v, dict) and 'f1' in v:
            return v['f1']
    return 0.0


def _get_per_label_f1(results: dict) -> dict:
    """Extract per-label F1 dict from opf eval output."""
    for key in ('per_label', 'by_label', 'labels', 'entities'):
        if key in results and isinstance(results[key], dict):
            label_data = results[key]
            return {
                label: (v['f1'] if isinstance(v, dict) else v)
                for label, v in label_data.items()
            }
    return {}


def run_opf_eval(dataset_path: str, checkpoint: str, label: str) -> dict:
    """Run opf eval and return parsed results dict."""
    cmd = ['opf', 'eval', dataset_path, '--checkpoint', checkpoint]
    print(f'[{label}] Command: {" ".join(cmd)}')
    res = subprocess.run(cmd, capture_output=True, text=True)

    if res.returncode != 0:
        print(f'[{label}] STDERR:\n{res.stderr}')
        raise RuntimeError(f'opf eval failed (code {res.returncode}) on {label}')

    stdout = res.stdout.strip()
    if stdout:
        try:
            return json.loads(stdout)
        except json.JSONDecodeError:
            pass

    for fname in ('eval_results.json', 'eval_summary.json', 'results.json'):
        candidate = os.path.join(checkpoint, fname)
        if os.path.exists(candidate):
            with open(candidate) as f:
                return json.load(f)

    print(f'[{label}] Raw stdout:\n{stdout[:3000]}')
    raise RuntimeError(
        f'Could not parse opf eval output for {label}. '
        'Inspect stdout above — check `opf eval --help` for the actual output format.'
    )


print('Eval helpers defined.')

In [ ]:
# Confirm opf eval CLI flags — if --checkpoint is wrong, fix run_opf_eval() above
help_res = subprocess.run(['opf', 'eval', '--help'], capture_output=True, text=True)
print(help_res.stdout or help_res.stderr)

In [ ]:
test_jsonl = str(DATA_DIR / 'test.jsonl')

print('Running full evaluation on test set...')
print('-' * 60)
full_results = run_opf_eval(test_jsonl, local_model, 'full')
print('\nFull eval results:')
print(json.dumps(full_results, indent=2))

In [ ]:
LANGUAGES = ['en', 'es', 'sv', 'de', 'it', 'nl', 'fr']
lang_results = {}

for lang in LANGUAGES:
    if lang not in lang_paths:
        print(f'[{lang}] No test examples found — skipping.')
        continue
    lang_results[lang] = run_opf_eval(lang_paths[lang], local_model, lang)
    print(f'[{lang}] F1: {_get_overall_f1(lang_results[lang]):.4f}\n')

print('Per-language evaluation complete.')

In [ ]:
overall_f1 = _get_overall_f1(full_results)
per_label_f1 = _get_per_label_f1(full_results)
lang_f1_vals = {lang: _get_overall_f1(res) for lang, res in lang_results.items()}

print(f'Overall entity F1: {overall_f1:.4f}')
print(f'\nPer-label F1 ({len(per_label_f1)} labels):')
for label, f1 in sorted(per_label_f1.items(), key=lambda x: -x[1]):
    print(f'  {label:40s} {f1:.4f}')

wandb.log({'eval/overall_f1': overall_f1})

if per_label_f1:
    label_table = wandb.Table(
        columns=['label', 'f1'],
        data=sorted(per_label_f1.items())
    )
    wandb.log({'eval/per_label_f1': label_table})

for lang, f1 in lang_f1_vals.items():
    wandb.log({f'eval/f1_{lang}': f1})

if lang_f1_vals:
    lang_table = wandb.Table(
        columns=['language', 'f1'],
        data=sorted(lang_f1_vals.items())
    )
    wandb.log({'eval/per_language_f1': lang_table})

print('Eval results logged to W&B.')

In [ ]:
FINTECH_LABELS = [
    'iban', 'bban', 'lei', 'isin', 'cusip',
    'swift_mt_ref', 'policy_number', 'vat_number', 'loan_number', 'crypto_address',
]
SHARED_FINANCIAL = [
    'account_number', 'bank_routing_number', 'credit_debit_card', 'swift_bic',
    'tax_id', 'ssn', 'national_id',
]
GENERAL_PII = [
    'first_name', 'last_name', 'email', 'phone_number', 'street_address',
    'date_of_birth', 'ipv4', 'url', 'user_name', 'password',
]

def print_f1_table(title, labels, f1_dict):
    print(f'**{title}:**')
    print('| Label | This model F1 |')
    print('|---|---|')
    for label in labels:
        f1 = f1_dict.get(label)
        val = f'{f1:.3f}' if f1 is not None else 'n/a'
        print(f'| `{label}` | {val} |')
    print()

lang_display = {
    'en': 'English (en)', 'es': 'Spanish (es)', 'sv': 'Swedish (sv)',
    'de': 'German (de)', 'it': 'Italian (it)', 'nl': 'Dutch (nl)', 'fr': 'French (fr)',
}

print('### Entity-level F1 — copy into README\n')
print_f1_table('Fintech labels (new)', FINTECH_LABELS, per_label_f1)
print_f1_table('Shared financial labels', SHARED_FINANCIAL, per_label_f1)
print_f1_table('General PII', GENERAL_PII, per_label_f1)

print('### Per-language F1 — copy into README\n')
print('| Language | This model F1 |')
print('|---|---|')
for lang in LANGUAGES:
    f1 = lang_f1_vals.get(lang)
    val = f'{f1:.3f}' if f1 is not None else 'n/a'
    print(f'| {lang_display.get(lang, lang)} | {val} |')

In [ ]:
wandb.finish()
print('W&B run complete. View at:', run.url)